In [204]:
import sys
import os

sys.path.append(os.path.abspath("../")) 

import torch
from torch import nn
from torch.utils.data import DataLoader
from blocks import CatDogDataset
from utils import get_detection_model, get_data_loaders
import json

In [205]:
def intersection_over_union(bbox_pred, bbox_target):
    bbox_pred_xmin = bbox_pred[..., 0:1] - (bbox_pred[..., 2:3] / 2)
    bbox_pred_ymin = bbox_pred[..., 1:2] - (bbox_pred[..., 3:4] / 2)
    bbox_pred_xmax = bbox_pred[..., 0:1] + (bbox_pred[..., 2:3] / 2)
    bbox_pred_ymax = bbox_pred[..., 1:2] + (bbox_pred[..., 3:4] / 2)

    bbox_target_xmin = bbox_target[..., 0:1] - (bbox_target[..., 2:3] / 2)
    bbox_target_ymin = bbox_target[..., 1:2] - (bbox_target[..., 3:4] / 2)
    bbox_target_xmax = bbox_target[..., 0:1] + (bbox_target[..., 2:3] / 2)
    bbox_target_ymax = bbox_target[..., 1:2] + (bbox_target[..., 3:4] / 2)

    bbox_inter_xmin = torch.max(bbox_pred_xmin, bbox_target_xmin)
    bbox_inter_ymin = torch.max(bbox_pred_ymin, bbox_target_ymin)
    bbox_inter_xmax = torch.min(bbox_pred_xmax, bbox_target_xmax)
    bbox_inter_ymax = torch.min(bbox_pred_ymax, bbox_target_ymax)

    width = (bbox_inter_xmax - bbox_inter_xmin).clamp(0)
    height = (bbox_inter_ymax - bbox_inter_ymin).clamp(0)

    intersection = width * height

    bbox_union_pred = torch.abs((bbox_pred_xmax - bbox_pred_xmin) * (bbox_pred_ymax - bbox_pred_ymin))
    bbox_union_target = torch.abs((bbox_target_xmax - bbox_target_xmin) * (bbox_target_ymax - bbox_target_ymin))

    union = bbox_union_pred + bbox_union_target - intersection

    iou = intersection / (union + 1e-6)

    return iou

In [206]:
boxes_preds = torch.tensor([[0.4, 0.4, 0.5, 0.5]])
boxes_labels = torch.tensor([[0.5, 0.5, 0.4, 0.4]])

In [207]:
intersection_over_union(boxes_preds, boxes_labels)

tensor([[0.4261]])

In [208]:
train_loader, val_loader, test_loader = get_data_loaders()
model = get_detection_model(num_classes=37)

In [209]:
image, target = next(iter(train_loader))
pred = torch.rand_like(target)

target.shape, pred.shape

(torch.Size([64, 7, 7, 47]), torch.Size([64, 7, 7, 47]))

Let's write loss for the first row:

In [210]:
exists_box = target[..., 4:5]

In [211]:
pred_box1 = pred[..., :4]
pred_box2 = pred[..., 5:9]
target_box = target[..., :4]


pred_box1.shape, pred_box2.shape, target_box.shape

(torch.Size([64, 7, 7, 4]),
 torch.Size([64, 7, 7, 4]),
 torch.Size([64, 7, 7, 4]))

In [212]:
pred_box1_iou = intersection_over_union(pred_box1, target_box)
pred_box2_iou = intersection_over_union(pred_box2, target_box)

pred_box1_iou.shape, pred_box2_iou.shape

box1_winner = (pred_box1_iou >= pred_box2_iou).to(int)
box2_winner = (pred_box1_iou < pred_box2_iou).to(int)

In [213]:
pred_x1_y1 = pred[..., :2]
pred_x2_y2 = pred[..., 5:7]
target_x_y = target[..., :2]

pred_x1_y1.shape, pred_x2_y2.shape, target_x_y.shape

(torch.Size([64, 7, 7, 2]),
 torch.Size([64, 7, 7, 2]),
 torch.Size([64, 7, 7, 2]))

In [214]:
box1_loss = torch.pow(pred_x1_y1 - target_x_y, 2)
box2_loss = torch.pow(pred_x2_y2 - target_x_y, 2)

final_x_y_loss = exists_box * ((box1_loss * box1_winner) + (box2_loss * box2_winner))


lambda_coord = 5
total_x_y_loss = lambda_coord * torch.sum(final_x_y_loss)

total_x_y_loss

tensor(86.9423)

Let's write loss for the second row:

In [215]:
exists_box = target[..., 4:5]

exists_box.shape

torch.Size([64, 7, 7, 1])

In [216]:
pred_box1 = pred[..., :4]
pred_box2 = pred[..., 5:9]
target_box = target[..., :4]


pred_box1.shape, pred_box2.shape, target_box.shape

(torch.Size([64, 7, 7, 4]),
 torch.Size([64, 7, 7, 4]),
 torch.Size([64, 7, 7, 4]))

In [217]:
pred_box1_iou = intersection_over_union(pred_box1, target_box)
pred_box2_iou = intersection_over_union(pred_box2, target_box)


box1_winner = (pred_box1_iou >= pred_box2_iou).to(int)
box2_winner = (pred_box1_iou < pred_box2_iou).to(int)

box1_winner.shape, box2_winner.shape

(torch.Size([64, 7, 7, 1]), torch.Size([64, 7, 7, 1]))

In [218]:
pred_w1_h1 = pred[..., 2:4]
pred_w2_h2 = pred[..., 7:9]

pred_w1_h1 = torch.sign(pred_w1_h1) * torch.sqrt(torch.abs(pred_w1_h1) + 1e-6)
pred_w2_h2 = torch.sign(pred_w2_h2) * torch.sqrt(torch.abs(pred_w2_h2) + 1e-6)


target_w_h = torch.sqrt(target[..., 2:4])

pred_w1_h1.shape, pred_w2_h2.shape, target_w_h.shape

(torch.Size([64, 7, 7, 2]),
 torch.Size([64, 7, 7, 2]),
 torch.Size([64, 7, 7, 2]))

In [219]:
box1_loss = torch.pow(target_w_h - pred_w1_h1, 2)
box2_loss = torch.pow(target_w_h - pred_w2_h2, 2)

box1_loss.shape, box2_loss.shape

(torch.Size([64, 7, 7, 2]), torch.Size([64, 7, 7, 2]))

In [220]:
final_w_h_loss = exists_box * (box1_loss * box1_winner + box2_loss * box2_winner)
total_w_h_loss = lambda_coord * torch.sum(final_w_h_loss)

total_w_h_loss

tensor(41.9099)

Let's write loss for the third row:

In [221]:
exists_box = target[..., 4:5]

exists_box.shape

torch.Size([64, 7, 7, 1])

In [222]:
pred_box1 = pred[..., :4]
pred_box2 = pred[..., 5:9]
target_box = target[..., :4]


pred_box1.shape, pred_box2.shape, target_box.shape

(torch.Size([64, 7, 7, 4]),
 torch.Size([64, 7, 7, 4]),
 torch.Size([64, 7, 7, 4]))

In [223]:
pred_box1_iou = intersection_over_union(pred_box1, target_box)
pred_box2_iou = intersection_over_union(pred_box2, target_box)


box1_winner = (pred_box1_iou >= pred_box2_iou).to(int)
box2_winner = (pred_box1_iou < pred_box2_iou).to(int)

box1_winner.shape, box2_winner.shape

(torch.Size([64, 7, 7, 1]), torch.Size([64, 7, 7, 1]))

In [224]:
pred_conf_1 = torch.sigmoid(pred[..., 4:5])
pred_conf_2 = torch.sigmoid(pred[..., 9:10])

target_conf_1 = pred_box1_iou.detach()
target_conf_2 = pred_box2_iou.detach()

pred_conf_1.shape, pred_conf_1.shape, target_conf_1.shape, target_conf_2.shape

(torch.Size([64, 7, 7, 1]),
 torch.Size([64, 7, 7, 1]),
 torch.Size([64, 7, 7, 1]),
 torch.Size([64, 7, 7, 1]))

In [225]:
loss_conf_1 = torch.pow(target_conf_1 - pred_conf_1, 2)
loss_conf_2 = torch.pow(target_conf_2 - pred_conf_2, 2)


final_loss_conf = exists_box * (loss_conf_1 * box1_winner + loss_conf_2 * box2_winner)

total_loss_conf = torch.sum(final_loss_conf)

total_loss_conf

tensor(17.2269)

Let's write loss for the fourth row:

In [226]:
lambda_noobj = 0.5

In [227]:
exists_box = target[..., 4:5]
no_box_exists = (target[..., 4:5] == 0).to(int)

exists_box.shape, no_box_exists.shape

(torch.Size([64, 7, 7, 1]), torch.Size([64, 7, 7, 1]))

In [228]:
pred_box1 = pred[..., :4]
pred_box2 = pred[..., 5:9]
target_box = target[..., :4]


pred_box1.shape, pred_box2.shape, target_box.shape

(torch.Size([64, 7, 7, 4]),
 torch.Size([64, 7, 7, 4]),
 torch.Size([64, 7, 7, 4]))

In [229]:
pred_box1_iou = intersection_over_union(pred_box1, target_box)
pred_box2_iou = intersection_over_union(pred_box2, target_box)


box1_winner = (pred_box1_iou >= pred_box2_iou).to(int)
box2_winner = (pred_box1_iou < pred_box2_iou).to(int)

box1_winner.shape, box2_winner.shape

(torch.Size([64, 7, 7, 1]), torch.Size([64, 7, 7, 1]))

In [230]:
pred_conf_1 = torch.sigmoid(pred[..., 4:5])
pred_conf_2 = torch.sigmoid(pred[..., 9:10])

target_conf_1 = 0.0
target_conf_2 = 0.0

pred_conf_1.shape, pred_conf_1.shape

(torch.Size([64, 7, 7, 1]), torch.Size([64, 7, 7, 1]))

In [231]:
loss_conf_1 = torch.pow(target_conf_1 - pred_conf_1, 2)
loss_conf_2 = torch.pow(target_conf_2 - pred_conf_2, 2)

final_loss_conf =                                                             \
    (no_box_exists * (loss_conf_1 + loss_conf_2))                             \
    +                                                                         \
    (exists_box * (loss_conf_1 * box2_winner + loss_conf_2 * box1_winner))

total_loss = lambda_noobj * torch.sum(final_loss_conf)

total_loss

tensor(1211.4524)

Let's write loss for the fifth row:

In [232]:
exists_box = target[..., 4:5]

exists_box.shape

torch.Size([64, 7, 7, 1])

In [233]:
pred_classes = torch.softmax(pred[..., 10:], dim=-1)
target_classes = target[..., 10:]

pred_classes.shape, target_classes.shape

(torch.Size([64, 7, 7, 37]), torch.Size([64, 7, 7, 37]))

In [234]:
loss_classes = torch.pow(target_classes - pred_classes , 2)

loss_classes.shape

torch.Size([64, 7, 7, 37])

In [235]:
loss_classes = exists_box * loss_classes

loss_classes.shape

torch.Size([64, 7, 7, 37])

In [236]:
total_loss_classes = torch.sum(loss_classes)

total_loss_classes

tensor(62.1915)

Lets summarize all of this into one loss function:

In [ ]:
class YoloLoss(nn.Module):
    def __init__(self, S=7, B=2, C=37):
        super(YoloLoss, self).__init__()
        self.S = S
        self.B = B
        self.C = C
        self.lambda_coord = 5.0
        self.lambda_noobj = 0.5

    def forward(self, preds, targets):
        exists_box = targets[..., 4:5]
        no_box_exists = 1 - exists_box

        preds_box1 = preds[..., 0:4]
        preds_box2 = preds[..., 5:9]
        targets_box = targets[..., 0:4]

        preds_box1_iou = intersection_over_union(preds_box1, targets_box)
        preds_box2_iou = intersection_over_union(preds_box2, targets_box)

        box1_winner = (preds_box1_iou >= preds_box2_iou).to(int)
        box2_winner = (preds_box1_iou < preds_box2_iou).to(int)

        preds_xy_1 = preds[..., 0:2]
        preds_xy_2 = preds[..., 5:7]
        targets_xy = targets[..., 0:2]

        loss_xy_1 = torch.pow(targets_xy - preds_xy_1, 2)
        loss_xy_2 = torch.pow(targets_xy - preds_xy_2, 2)

        final_xy_loss = exists_box * ((loss_xy_1 * box1_winner) + (loss_xy_2 * box2_winner))
        total_xy_loss = self.lambda_coord * torch.sum(final_xy_loss)

        preds_wh_1 = torch.sign(preds[..., 2:4]) * torch.sqrt(torch.abs(preds[..., 2:4]) + 1e-6)
        preds_wh_2 = torch.sign(preds[..., 7:9]) * torch.sqrt(torch.abs(preds[..., 7:9]) + 1e-6)
        targets_wh = torch.sqrt(targets[..., 2:4])

        loss_wh_1 = torch.pow(targets_wh - preds_wh_1, 2)
        loss_wh_2 = torch.pow(targets_wh - preds_wh_2, 2)

        final_wh_loss = exists_box * ((loss_wh_1 * box1_winner) + (loss_wh_2 * box2_winner))
        total_wh_loss = self.lambda_coord * torch.sum(final_wh_loss)

        preds_conf_1 = torch.sigmoid(preds[..., 4:5])
        preds_conf_2 = torch.sigmoid(preds[..., 9:10])
        
        targets_conf_1 = preds_box1_iou.detach()
        targets_conf_2 = preds_box2_iou.detach()

        loss_obj_1 = torch.pow(targets_conf_1 - preds_conf_1, 2)
        loss_obj_2 = torch.pow(targets_conf_2 - preds_conf_2, 2)

        final_obj_loss = exists_box * ((loss_obj_1 * box1_winner) + (loss_obj_2 * box2_winner))
        total_obj_loss = torch.sum(final_obj_loss)


        loss_noobj_1 = torch.pow(0.0 - preds_conf_1, 2)
        loss_noobj_2 = torch.pow(0.0 - preds_conf_2, 2)

        final_noobj_loss = (
            (no_box_exists * (loss_noobj_1 + loss_noobj_2)) + 
            (exists_box * ((loss_noobj_1 * box2_winner) + (loss_noobj_2 * box1_winner)))
        )
        total_noobj_loss = self.lambda_noobj * torch.sum(final_noobj_loss)

        preds_classes = torch.softmax(preds[..., 10:], dim=-1)
        targets_classes = targets[..., 10:]

        loss_classes = torch.pow(targets_classes - preds_classes, 2)

        final_class_loss = exists_box * loss_classes
        total_class_loss = torch.sum(final_class_loss)


        total_loss = (
            total_xy_loss + 
            total_wh_loss + 
            total_obj_loss + 
            total_noobj_loss + 
            total_class_loss
        )

        return total_loss